# Часть 1. Загрузка и предобработка данных

В этом ноутбуке мы загружаем исходный датасет `NetflixShows`, изучаем его структуру, устраняем проблемы качества данных, такие как дубликаты, пропуски и избыточные признаки.

Результат этой части - это очищенный датасет `netflix_clean_1.csv`, который используется в последующих частях анализа.

#### Выполнил Ефимов Алексей


## Шаг 1. Загрузка и первичный осмотр данных

Читаем файл с явным указанием кодировки `utf-8-sig`, чтобы избежать артефакта BOM-метки
(`\xef\xbb\xbf`), который появляется при пересохранении файла из Excel в CSV.


In [ ]:
import pandas as pd

data = pd.read_csv("NetflixShows.csv", encoding="utf-8-sig", sep=";")
data.head(3)

Представленный датасет содержит следующие признаки (колонки):

- `title` - название фильма, сериала
- `rating` - возрастной рейтинг
- `ratingLevel` - детализация, описание возрастного рейтинга
- `ratingDescription` - числовая кодировка рейтинга
- `release year` - дата выхода фильма или сериала
- `user rating score` - оценка контента пользователями
- `user rating size` - количество пользователей, которые поставили оценку

Признак `ratingDescription` - это просто числовая кодировка `rating`, которая у нас уже есть в текстовом виде. Он не несёт новой информации, поэтому удаляем его сразу.


In [ ]:
del data["ratingDescription"]
data.head(3)

## Шаг 2. Дубликаты

При рассмотрении данных заметно, что одни и те же шоу встречаются несколько раз.
Возможно данные были собраны парсингом или несколькими участниками, либо склейка файлов без проверки.


In [ ]:
data[data['title'].str.lower() == 'black mirror']

In [ ]:
data[data.duplicated()].sort_values("title").head(20)

Всего строк в файле 1000. Из них дублей - 500. Ровно половина датасета это дубликаты.


In [ ]:
print(data.shape[0])
print(data.duplicated().sum())

Применяем `drop_duplicates()`, потому что задублированы именно целые строки. После у нас остается 500 уникальных строк.


In [ ]:
data_no_duplicates = data.drop_duplicates()
data_no_duplicates.shape[0]

Также необходимо сбросить индекс и переименовываем датафрейм, чтобы не путаться. Дальше работает с датасетом `data_clear`


In [ ]:
data_no_duplicates.tail(5)

In [ ]:
data_clear = data_no_duplicates.copy().reset_index(drop=True)
data_clear.tail(5)

## Шаг 3. Пропуски (NaN)

In [ ]:
data_clear.isnull().sum()

Два столбца содержат пропуски:

- `ratingLevel` - текстовое описание рейтинга. Здесь 33 пропуска. Поскольку описание всегда одинаково для одного и того же `rating`, мы можем восстановить эти значения по уже имеющимся строкам.
- `user rating score` - оценка пользователей. Здесь уже 244 пропуска. Из 500 строк это очень много. Удалить или исключить из анализа столько наблюдений нельзя, поэтому заполним медианой по группам рейтинга.


### 3.1 Заполнение ratingLevel

Сделаем словарь `{rating: ratingLevel}` на основе строк, где значение  `ratingLevel` уже есть. И выполним замену пропусков.


In [ ]:
rating_to_level = (
    data_clear[data_clear["ratingLevel"].notna()]
    .groupby("rating")["ratingLevel"]
    .first()
    .to_dict()
)

data_clear["ratingLevel"] = data_clear["ratingLevel"].fillna(
    data_clear["rating"].map(rating_to_level)
)

data_clear["ratingLevel"].isnull().sum()

### 3.2 Заполнение user rating score

Заполняем медианой пропуски внутри каждой рейтинговой группы. Медиана устойчива к выбросам, здесь является типичной оценкой аудитории для данной категории контента.


In [ ]:
data_clear["user rating score"] = data_clear.groupby("rating")[
    "user rating score"
].transform(lambda x: x.fillna(x.median()))

data_clear["user rating score"].isna().sum()

Один пропуск остался - посмотрим, что это за строка.


In [ ]:
data_clear[data_clear["user rating score"].isna()]

Это шоу с рейтингом `UR` является единственной записью в своей группе, поэтому медиана не рассчиталась и не применилась замена.
Дропаем эту строку без сожалений.


In [ ]:
id_to_drop = data_clear[data_clear["user rating score"].isna()].index[0]
data_clear = data_clear.drop(id_to_drop).reset_index(drop=True)
data_clear["user rating score"].isna().sum()

### 3.3 Формат названий колонок (признаков)

Приведем названия признаков к единому формату. Выбрали snake_case

In [ ]:
data_clear = data_clear.rename(
    columns={
        "ratingLevel": "rating_level",
        "release year": "release_year",
        "user rating score": "rating_score",
        "user rating size": "rating_size",
    }
)
data_clear.columns.tolist()

### 3.4 Финальная проверка


In [ ]:
data_clear.isna().sum()

## Шаг 4. Сохранение результата

Сохраняем очищенный датасет. Он будет использоваться далее для обогащения данными из внешних источников.

In [ ]:
data_clear.to_csv("netflix_clean_1.csv", index=False)


## Итоги предобработки

Итоговый датасет: **499 строк, 6 признаков**, пропусков нет.


In [ ]:
data_clear.shape